# Training Client Side Model

## Starting Up

In [1]:
import json
import os
from glob import glob

import boto3
import kagglehub
import pandas as pd
from catboost import CatBoostClassifier
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

/home/sado/Projects/nexgen-example/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(
    override=True,
)

True

In [3]:
path = kagglehub.dataset_download(
    "rabieelkharoua/predict-online-gaming-behavior-dataset",
    output_dir="online-gaming-behavior-dataset",
)

print("Path to dataset files:", path)

Path to dataset files: online-gaming-behavior-dataset


In [4]:
file_paths = glob(f"{path}/*")
file_paths

['online-gaming-behavior-dataset/online_gaming_behavior_dataset.csv']

In [5]:
df = pd.read_csv(file_paths[0])

In [6]:
df.head()

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,9000,43,Male,Other,Strategy,16.271119,0,Medium,6,108,79,25,Medium
1,9001,29,Female,USA,Strategy,5.525961,0,Medium,5,144,11,10,Medium
2,9002,22,Female,USA,Sports,8.223755,0,Easy,16,142,35,41,High
3,9003,35,Male,USA,Action,5.265351,1,Easy,9,85,57,47,Medium
4,9004,33,Male,Europe,Action,15.531945,0,Medium,2,131,95,37,Medium


In [7]:
df.iloc[:10, :].to_csv("../resources/client_side_example.csv", index=False)
df = df.iloc[10:, :]

In [ ]:
df["EngagementLevel"].value_counts()

## Formatting The Data

In [20]:
def multi_shot_encoding(
    unique_values: list,
    save_file: str,
) -> dict[any, int]:
    """It applies the multi shot encoding."""
    mapping = {}
    rev_mapping = {}

    for i in range(len(unique_values)):
        mapping[unique_values[i]] = i
        rev_mapping[i] = unique_values[i]

    with open(save_file, "w") as f:
        json.dump(
            mapping,
            f,
            indent=4,
        )

    return mapping

In [21]:
gender_map = multi_shot_encoding(
    unique_values=df["Gender"].unique(),
    save_file="../resources/gender_map.json",
)

df["Gender"] = df["Gender"].map(gender_map)

In [22]:
location_map = multi_shot_encoding(
    unique_values=df["Location"].unique(),
    save_file="../resources/location_map.json",
)

df["Location"] = df["Location"].map(location_map)

In [23]:
game_genre_map = multi_shot_encoding(
    unique_values=df["GameGenre"].unique(),
    save_file="../resources/game_genre_map.json",
)

df["GameGenre"] = df["GameGenre"].map(game_genre_map)

In [24]:
game_difficulty_map = multi_shot_encoding(
    unique_values=df["GameDifficulty"].unique(),
    save_file="../resources/game_difficulty_map.json",
)

df["GameDifficulty"] = df["GameDifficulty"].map(game_difficulty_map)

In [25]:
engagement_level_map = multi_shot_encoding(
    unique_values=df["EngagementLevel"].unique(),
    save_file="../resources/engagement_level_map.json",
)

df["EngagementLevel"] = df["EngagementLevel"].map(engagement_level_map)

In [27]:
del df["PlayerID"]

In [28]:
df.head()

,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
10,17,0,0,0,4.829916,0,0,8,95,14,12,0
11,36,1,1,1,5.535981,1,1,16,124,62,31,0
12,16,0,0,2,18.776234,1,1,9,18,52,32,0
13,38,1,0,0,8.701959,0,1,0,156,33,47,1
14,44,0,0,1,17.975200,0,1,8,41,98,1,1


## Training The Model

In [29]:
y = df["EngagementLevel"].values
del df["EngagementLevel"]
X = df.values
del df

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
)

In [33]:
model = CatBoostClassifier(verbose=False)

In [34]:
model.fit(X_train, y_train)

CatBoostClassifier(verbose=False)

## Prediction

In [36]:
y_pred = model.predict(X_test)

In [37]:
acc = accuracy_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")

In [38]:
print(f"Accuracy Score: {acc}")
print(f"MCC Score: {mcc}")
print(f"Precision Score: {precision}")
print(f"Recall Score: {recall}")
print(f"F1 Score: {f1}")

Accuracy Score: 0.9183010618363523
MCC Score: 0.8703966812880447
Precision Score: 0.9188306931025401
Recall Score: 0.9183010618363523
F1 Score: 0.9179808763139286


# Saving Objects

In [42]:
minio_client = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY"),
    region_name=os.getenv("MINIO_REGION"),
)

In [43]:
bucket_name = os.getenv("MINIO_BUCKET", "nexgen-example")

existing_buckets = [b["Name"] for b in minio_client.list_buckets()["Buckets"]]
if bucket_name not in existing_buckets:
    minio_client.create_bucket(Bucket=bucket_name)

In [44]:
model_key = os.getenv("MINIO_MODEL_KEY")
model_path = "../resources/model.cbm"

model.save_model(model_path)
minio_client.upload_file(model_path, bucket_name, model_key)

print(f"Model saved to s3://{bucket_name}/{model_key}")

Model saved to s3://nexgen-example/catboost_model.cbm


In [45]:
minio_client.upload_file(
    "../resources/engagement_level_map.json",
    bucket_name,
    "engagement_level_map.json",
)

In [46]:
minio_client.upload_file(
    "../resources/game_difficulty_map.json",
    bucket_name,
    "game_difficulty_map.json",
)

In [47]:
minio_client.upload_file(
    "../resources/game_genre_map.json",
    bucket_name,
    "game_genre_map.json",
)

In [48]:
minio_client.upload_file(
    "../resources/gender_map.json",
    bucket_name,
    "gender_map.json",
)

In [49]:
minio_client.upload_file(
    "../resources/location_map.json",
    bucket_name,
    "location_map.json",
)